# 07 · Aligned validation comparison at T-60 minutes

This notebook implements only step 1: compare the frozen route+airline historical baseline
and the six notebook-06 model configurations on exactly the same deterministic 5% validation
sample. It never reads test and does not train a new machine-learning model.

Operational contract: the prediction is issued 60 minutes before scheduled off-block. Static
schedule and historical train-period aggregates are available. Outcomes from earlier flights
may be used in future work only when they were already observed by T-60; step 1 does not add
those rolling features.

The selection score gives equal weight to global MAE and MAE for flights whose realised delay
exceeds 15 minutes. The segment is diagnostic: its true label is not known at prediction time.


In [1]:
from pathlib import Path
import gc
import importlib.util
import math
import sys
import time

import numpy as np
import pandas as pd
import psutil
from pyspark import StorageLevel
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spark_flight_pipeline import create_spark

RUN_COMPARISON = True
PREDICTION_LEAD_MINUTES = 60
CATBOOST_TRAIN_SAMPLE_PERCENTS = (1, 5, 10)
CATBOOST_WEIGHT_SWEEP_GLOBAL_SHARES = (1.0, 0.75, 0.65, 0.5)
CATBOOST_GLOBAL_MAE_GUARDRAIL_MINUTES = 1.0
CATBOOST_ITERATIONS = 600
VALIDATION_SAMPLE_PERCENT = 5
GLOBAL_MAE_WEIGHT = 0.5
DELAYED_MAE_WEIGHT = 0.5
MIN_AVAILABLE_RAM_GB = 5.0
TARGET = "Arrival_Delay_Min"
ROUTE_MIN_ROWS = 100
ROUTE_AIRLINE_MIN_ROWS = 20  # Frozen in notebook 05; not retuned here.
EXPECTED_TRAIN_ROWS = 2_457_169
EXPECTED_VALIDATION_ROWS = 591_391
EXPECTED_VALIDATION_SAMPLE_ROWS = 29_315
DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "model" / "arrival_pre"
LEGACY_REPORT_ROOT = PROJECT_ROOT / "reports" / "modeling" / "legacy"
MODEL_REPORT_PATH = LEGACY_REPORT_ROOT / "benchmark" / "06_model_benchmark.csv"
REPORT_PATH = LEGACY_REPORT_ROOT / "aligned_validation" / "07_aligned_validation_comparison.csv"
WEIGHT_SWEEP_REPORT_PATH = REPORT_PATH.parent / "07_catboost_weight_sweep.csv"
LEARNING_CURVE_REPORT_PATH = REPORT_PATH.parent / "07_catboost_learning_curve.csv"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
CATBOOST_MODEL_PATH = PROJECT_ROOT / "models" / "07_catboost_t60_selected_10pct.cbm"
FORBIDDEN_COLUMNS = {
    "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
    "Actual Distance Flown (nm)", "Departure_Delay_Min",
}


## Preflight

Close memory-heavy applications before running. The comparison recomputes historical
aggregates from full train but keeps intermediate frames on disk where appropriate.


In [2]:
memory = psutil.virtual_memory()
available_ram_gb = memory.available / (1024 ** 3)
print({
    "run_enabled": RUN_COMPARISON,
    "prediction_lead_minutes": PREDICTION_LEAD_MINUTES,
    "available_ram_gb": round(available_ram_gb, 2),
    "required_available_ram_gb": MIN_AVAILABLE_RAM_GB,
    "global_mae_weight": GLOBAL_MAE_WEIGHT,
    "delayed_mae_weight": DELAYED_MAE_WEIGHT,
})
assert abs(GLOBAL_MAE_WEIGHT + DELAYED_MAE_WEIGHT - 1.0) < 1e-12
if RUN_COMPARISON and importlib.util.find_spec("catboost") is None:
    raise RuntimeError(
        'CatBoost is missing. From the project root run: '
        '.\\.venv313\\Scripts\\python.exe -m pip install catboost'
    )
if RUN_COMPARISON and available_ram_gb < MIN_AVAILABLE_RAM_GB:
    raise RuntimeError(
        f"Only {available_ram_gb:.2f} GB RAM is available; retry with at least "
        f"{MIN_AVAILABLE_RAM_GB:.1f} GB after imports."
    )
assert MODEL_REPORT_PATH.exists(), MODEL_REPORT_PATH


{'run_enabled': True, 'prediction_lead_minutes': 60, 'available_ram_gb': 5.27, 'required_available_ram_gb': 5.0, 'global_mae_weight': 0.5, 'delayed_mae_weight': 0.5}


## Load only train and validation

The deterministic hash expression is identical to notebook 06. Test remains untouched.


In [3]:
def deterministic_percent_sample(frame, percent):
    return frame.filter(F.pmod(F.hash("ECTRL ID"), F.lit(100)) < percent)

if RUN_COMPARISON:
    spark = create_spark(
        "arrival-pre-aligned-validation",
        master="local[2]", driver_memory="4g", shuffle_partitions=8,
    )
    spark.sparkContext.setLogLevel("WARN")
    train = spark.read.parquet(str(DATA_ROOT / "train"))
    validation = spark.read.parquet(str(DATA_ROOT / "validation"))
    counts = {"train": train.count(), "validation": validation.count()}
    assert counts == {
        "train": EXPECTED_TRAIN_ROWS,
        "validation": EXPECTED_VALIDATION_ROWS,
    }, counts
    assert train.schema.json() == validation.schema.json()
    assert TARGET in train.columns
    assert not (FORBIDDEN_COLUMNS & set(train.columns))
    catboost_source_columns = [
        "ECTRL ID", "ADEP", "ADES", "AC Operator", "AC Type_grouped",
        "STATFOR Market Segment", "Class_aircraft",
        "Number+Engine Type_aircraft", "Requested_FL_Imputed",
        "FILED OFF BLOCK TIME", "FILED ARRIVAL TIME", TARGET,
    ]
    assert not (set(catboost_source_columns) - set(train.columns))
    max_train_percent = max(CATBOOST_TRAIN_SAMPLE_PERCENTS)
    catboost_train_max_sample = (
        train.withColumn(
            "_sample_bucket", F.pmod(F.hash("ECTRL ID"), F.lit(100))
        )
        .filter(F.col("_sample_bucket") < max_train_percent)
        .select(*catboost_source_columns, "_sample_bucket")
    )
    catboost_validation_sample = deterministic_percent_sample(
        validation, VALIDATION_SAMPLE_PERCENT
    ).select(*catboost_source_columns)
    validation_sample = catboost_validation_sample.select(
        "ECTRL ID", "ADEP", "ADES", "AC Operator", TARGET
    )
    catboost_train_max_sample_rows = catboost_train_max_sample.count()
    validation_sample_rows = validation_sample.count()
    assert 240_000 < catboost_train_max_sample_rows < 255_000
    assert validation_sample_rows == EXPECTED_VALIDATION_SAMPLE_ROWS
    print({
        "full_counts": counts,
        "catboost_train_max_percent": max_train_percent,
        "catboost_train_max_sample_rows": catboost_train_max_sample_rows,
        "validation_sample_rows": validation_sample_rows,
    })
else:
    print("Comparison disabled.")


{'full_counts': {'train': 2457169, 'validation': 591391}, 'catboost_train_max_percent': 10, 'catboost_train_max_sample_rows': 245590, 'validation_sample_rows': 29315}


## Fit the frozen historical baseline using train only

Fallback order: route+airline (minimum 20), route (minimum 100), departure
airport+airline (minimum 20), departure airport (minimum 100), global median.
The thresholds are inherited from notebook 05 and are not selected again on this sample.


In [4]:
if RUN_COMPARISON:
    global_median = train.agg(
        F.percentile_approx(TARGET, 0.5, 10_000).alias("global_median")
    ).first()["global_median"]
    route_stats = (
        train.groupBy("ADEP", "ADES")
        .agg(F.count("*").alias("route_rows"),
             F.percentile_approx(TARGET, 0.5, 10_000).alias("route_median"))
        .filter(F.col("route_rows") >= ROUTE_MIN_ROWS)
    )
    adep_stats = (
        train.groupBy("ADEP")
        .agg(F.count("*").alias("adep_rows"),
             F.percentile_approx(TARGET, 0.5, 10_000).alias("adep_median"))
        .filter(F.col("adep_rows") >= ROUTE_MIN_ROWS)
    )
    route_airline_stats = (
        train.groupBy("ADEP", "ADES", "AC Operator")
        .agg(F.count("*").alias("route_airline_rows"),
             F.percentile_approx(TARGET, 0.5, 10_000).alias("route_airline_median"))
        .filter(F.col("route_airline_rows") >= ROUTE_AIRLINE_MIN_ROWS)
    )
    adep_airline_stats = (
        train.groupBy("ADEP", "AC Operator")
        .agg(F.count("*").alias("adep_airline_rows"),
             F.percentile_approx(TARGET, 0.5, 10_000).alias("adep_airline_median"))
        .filter(F.col("adep_airline_rows") >= ROUTE_AIRLINE_MIN_ROWS)
    )
    print({
        "global_median": global_median,
        "eligible_routes": route_stats.count(),
        "eligible_departure_airports": adep_stats.count(),
        "eligible_route_airlines": route_airline_stats.count(),
        "eligible_departure_airport_airlines": adep_airline_stats.count(),
    })


{'global_median': 2.4166666666666665, 'eligible_routes': 6337, 'eligible_departure_airports': 722, 'eligible_route_airlines': 21755, 'eligible_departure_airport_airlines': 6263}


In [5]:
def add_historical_prediction(frame):
    return (
        frame.join(F.broadcast(route_airline_stats),
                   ["ADEP", "ADES", "AC Operator"], "left")
        .join(F.broadcast(route_stats), ["ADEP", "ADES"], "left")
        .join(F.broadcast(adep_airline_stats), ["ADEP", "AC Operator"], "left")
        .join(F.broadcast(adep_stats), ["ADEP"], "left")
        .withColumn(
            "fallback_source",
            F.when(F.col("route_airline_median").isNotNull(), "route_airline")
            .when(F.col("route_median").isNotNull(), "route")
            .when(F.col("adep_airline_median").isNotNull(), "departure_airport_airline")
            .when(F.col("adep_median").isNotNull(), "departure_airport")
            .otherwise("global"),
        )
        .withColumn(
            "prediction",
            F.coalesce("route_airline_median", "route_median",
                       "adep_airline_median", "adep_median", F.lit(global_median)),
        )
    )

if RUN_COMPARISON:
    baseline_predictions = add_historical_prediction(validation_sample).persist(
        StorageLevel.DISK_ONLY
    )
    assert baseline_predictions.count() == EXPECTED_VALIDATION_SAMPLE_ROWS
    baseline_predictions.groupBy("fallback_source").count().orderBy(
        F.desc("count")
    ).show(truncate=False)


+-------------------------+-----+
|fallback_source          |count|
+-------------------------+-----+
|route_airline            |27364|
|departure_airport_airline|1129 |
|route                    |522  |
|departure_airport        |253  |
|global                   |47   |
+-------------------------+-----+



## Score the baseline on the identical validation rows


In [6]:
def collect_segment_metrics(predictions):
    errors = (
        predictions.select(TARGET, "prediction")
        .withColumn("absolute_error", F.abs(F.col(TARGET) - F.col("prediction")))
        .withColumn("squared_error", F.pow(F.col(TARGET) - F.col("prediction"), 2))
    )
    exclusive = errors.withColumn(
        "segment",
        F.when(F.col(TARGET) <= 15, "punctual_<=15")
        .when(F.col(TARGET) <= 60, "moderate_15_60")
        .otherwise("severe_>60"),
    )
    segmented = (
        exclusive.unionByName(errors.withColumn("segment", F.lit("all")))
        .unionByName(errors.filter(F.col(TARGET) > 15).withColumn(
            "segment", F.lit("delayed_>15")
        ))
    )
    rows = (
        segmented.groupBy("segment")
        .agg(
            F.count("*").alias("rows"),
            F.avg("absolute_error").alias("MAE"),
            F.sqrt(F.avg("squared_error")).alias("RMSE"),
            F.percentile_approx("absolute_error", 0.5, 10_000).alias(
                "median_absolute_error"
            ),
            F.percentile_approx("absolute_error", 0.9, 10_000).alias(
                "p90_absolute_error"
            ),
        ).collect()
    )
    return pd.DataFrame([row.asDict() for row in rows])

if RUN_COMPARISON:
    baseline_metrics_pd = collect_segment_metrics(baseline_predictions)
    baseline_metrics_pd["model"] = "historical_route_airline_fallback"
    baseline_metrics_pd["numeric_variant"] = "not_applicable"
    baseline_metrics_pd["status"] = "ok"
    baseline_metrics_pd["training_scope"] = "full_train"
    display(baseline_metrics_pd.sort_values("segment"))


,segment,rows,MAE,RMSE,median_absolute_error,p90_absolute_error,model,numeric_variant,status,training_scope
3,all,29315,9.938292,15.270555,7.116667,20.450000,historical_route_airline_fallback,not_applicable,ok,full_train
4,delayed_>15,5668,21.108836,28.662692,16.683333,39.200000,historical_route_airline_fallback,not_applicable,ok,full_train
2,moderate_15_60,5343,17.854776,21.004169,15.983333,32.866667,historical_route_airline_fallback,not_applicable,ok,full_train
0,punctual_<=15,23647,7.260800,9.600238,5.900000,14.900000,historical_route_airline_fallback,not_applicable,ok,full_train
1,severe_>60,325,74.605590,84.112494,64.666667,117.883333,historical_route_airline_fallback,not_applicable,ok,full_train


## CatBoost: weight sweep and 1/5/10% learning curve

Airports, operator and aircraft type are passed as native categories. Every training subset is
nested and deterministic, sorted chronologically, and fitted with `has_time=True`. Validation is
fixed, so differences across experiments come only from weighting or added training rows.

First, four loss mixtures are compared on 1% train. A global share `g` maps to delayed-row
weight `1 + ((1-g)/g) * N/N_delayed`; `g=1` is ordinary MAE and `g=0.5` is the previous
aggressive weighting. Selection minimizes the agreed combined MAE, but rejects configurations
whose global MAE is over one minute worse than the best notebook-06 model. The selected mixture
is then trained on 1%, 5% and 10% to distinguish model limitations from lack of training data.


In [7]:
CATBOOST_CATEGORICAL_COLUMNS = [
    "ADEP", "ADES", "AC Operator", "AC Type_grouped",
    "STATFOR Market Segment", "Class_aircraft",
    "Number+Engine Type_aircraft",
]
CATBOOST_NUMERIC_COLUMNS = [
    "Requested_FL_Imputed", "scheduled_duration_min",
    "departure_hour_sin", "departure_hour_cos",
    "departure_dow_sin", "departure_dow_cos", "departure_month",
]

def prepare_catboost_pandas(frame):
    result = frame.toPandas()
    result["FILED OFF BLOCK TIME"] = pd.to_datetime(result["FILED OFF BLOCK TIME"])
    result["FILED ARRIVAL TIME"] = pd.to_datetime(result["FILED ARRIVAL TIME"])
    result = result.sort_values("FILED OFF BLOCK TIME").reset_index(drop=True)
    result["scheduled_duration_min"] = (
        result["FILED ARRIVAL TIME"] - result["FILED OFF BLOCK TIME"]
    ).dt.total_seconds() / 60.0
    hours = (result["FILED OFF BLOCK TIME"].dt.hour
             + result["FILED OFF BLOCK TIME"].dt.minute / 60.0)
    day_of_week = result["FILED OFF BLOCK TIME"].dt.dayofweek.astype(float)
    result["departure_hour_sin"] = np.sin(2 * math.pi * hours / 24.0)
    result["departure_hour_cos"] = np.cos(2 * math.pi * hours / 24.0)
    result["departure_dow_sin"] = np.sin(2 * math.pi * day_of_week / 7.0)
    result["departure_dow_cos"] = np.cos(2 * math.pi * day_of_week / 7.0)
    result["departure_month"] = result["FILED OFF BLOCK TIME"].dt.month.astype(float)
    for column in CATBOOST_CATEGORICAL_COLUMNS:
        result[column] = result[column].fillna("__MISSING__").astype(str)
    return result

def delayed_row_weight(global_share, row_count, delayed_count):
    assert 0 < global_share <= 1
    return 1.0 + ((1.0 - global_share) / global_share) * row_count / delayed_count

if RUN_COMPARISON:
    from catboost import CatBoostRegressor, Pool

    catboost_train_max_pd = prepare_catboost_pandas(catboost_train_max_sample)
    catboost_validation_pd_raw = prepare_catboost_pandas(catboost_validation_sample)
    catboost_features = CATBOOST_CATEGORICAL_COLUMNS + CATBOOST_NUMERIC_COLUMNS
    reference_metrics_pd = pd.read_csv(MODEL_REPORT_PATH)
    reference_global_mae = reference_metrics_pd.loc[
        (reference_metrics_pd["status"] == "ok")
        & (reference_metrics_pd["segment"] == "all"), "MAE"
    ].min()
    global_mae_limit = reference_global_mae + CATBOOST_GLOBAL_MAE_GUARDRAIL_MINUTES

    def fit_catboost_experiment(train_percent, global_share):
        train_pd = catboost_train_max_pd.loc[
            catboost_train_max_pd["_sample_bucket"] < train_percent
        ].copy()
        validation_pd = catboost_validation_pd_raw.copy()
        numeric_medians = train_pd[CATBOOST_NUMERIC_COLUMNS].median()
        assert not numeric_medians.isna().any(), numeric_medians
        train_pd[CATBOOST_NUMERIC_COLUMNS] = (
            train_pd[CATBOOST_NUMERIC_COLUMNS].fillna(numeric_medians)
        )
        validation_pd[CATBOOST_NUMERIC_COLUMNS] = (
            validation_pd[CATBOOST_NUMERIC_COLUMNS].fillna(numeric_medians)
        )
        y_train = train_pd[TARGET].astype(float)
        y_validation = validation_pd[TARGET].astype(float)
        delayed_train_count = int((y_train > 15).sum())
        delayed_validation_count = int((y_validation > 15).sum())
        train_delayed_weight = delayed_row_weight(
            global_share, len(y_train), delayed_train_count
        )
        validation_delayed_weight = delayed_row_weight(
            global_share, len(y_validation), delayed_validation_count
        )
        train_pool = Pool(
            train_pd[catboost_features], label=y_train,
            cat_features=CATBOOST_CATEGORICAL_COLUMNS,
            weight=np.where(y_train > 15, train_delayed_weight, 1.0),
        )
        validation_pool = Pool(
            validation_pd[catboost_features], label=y_validation,
            cat_features=CATBOOST_CATEGORICAL_COLUMNS,
            weight=np.where(y_validation > 15, validation_delayed_weight, 1.0),
        )
        model = CatBoostRegressor(
            loss_function="MAE", eval_metric="MAE",
            iterations=CATBOOST_ITERATIONS, learning_rate=0.05, depth=7,
            l2_leaf_reg=8, boosting_type="Plain", has_time=True,
            one_hot_max_size=10, max_ctr_complexity=2, random_seed=42,
            thread_count=4, od_type="Iter", od_wait=50,
            allow_writing_files=False, verbose=False,
        )
        fit_started = time.perf_counter()
        model.fit(train_pool, eval_set=validation_pool, use_best_model=True)
        fit_seconds = time.perf_counter() - fit_started
        predictions = model.predict(validation_pool)
        prediction_frame = spark.createDataFrame(pd.DataFrame({
            TARGET: y_validation.to_numpy(), "prediction": predictions,
        }))
        metrics = collect_segment_metrics(prediction_frame)
        model_name = f"catboost_g{global_share:.2f}_p{train_percent:02d}"
        metrics["model"] = model_name
        metrics["numeric_variant"] = "native_categorical_t60"
        metrics["status"] = "ok"
        metrics["training_scope"] = f"deterministic_{train_percent}pct_train"
        by_segment = metrics.set_index("segment")
        summary = {
            "model": model_name, "train_percent": train_percent,
            "train_rows": len(train_pd), "global_share": global_share,
            "delayed_row_weight": train_delayed_weight,
            "best_iteration": model.get_best_iteration(),
            "fit_seconds": fit_seconds,
            "global_MAE": by_segment.loc["all", "MAE"],
            "delayed_MAE": by_segment.loc["delayed_>15", "MAE"],
            "punctual_MAE": by_segment.loc["punctual_<=15", "MAE"],
            "severe_MAE": by_segment.loc["severe_>60", "MAE"],
        }
        summary["combined_MAE_score"] = (
            GLOBAL_MAE_WEIGHT * summary["global_MAE"]
            + DELAYED_MAE_WEIGHT * summary["delayed_MAE"]
        )
        del train_pd, validation_pd, train_pool, validation_pool, prediction_frame
        gc.collect()
        return model, metrics, summary

    experiment_metrics = []
    sweep_summaries = []
    sweep_models = {}
    for global_share in CATBOOST_WEIGHT_SWEEP_GLOBAL_SHARES:
        model, metrics, summary = fit_catboost_experiment(1, global_share)
        sweep_models[global_share] = model
        experiment_metrics.append(metrics)
        sweep_summaries.append(summary)
        print({key: round(value, 4) if isinstance(value, float) else value
               for key, value in summary.items()})

    weight_sweep_pd = pd.DataFrame(sweep_summaries).sort_values("global_share", ascending=False)
    weight_sweep_pd["passes_global_guardrail"] = (
        weight_sweep_pd["global_MAE"] <= global_mae_limit
    )
    eligible_weights_pd = weight_sweep_pd[weight_sweep_pd["passes_global_guardrail"]]
    assert not eligible_weights_pd.empty
    selected_weight_row = eligible_weights_pd.sort_values(
        ["combined_MAE_score", "global_MAE"]
    ).iloc[0]
    selected_global_share = float(selected_weight_row["global_share"])
    WEIGHT_SWEEP_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    weight_sweep_pd.to_csv(WEIGHT_SWEEP_REPORT_PATH, index=False)
    display(weight_sweep_pd)
    print({"reference_global_MAE": round(reference_global_mae, 4),
           "global_MAE_limit": round(global_mae_limit, 4),
           "selected_global_share": selected_global_share})

    learning_summaries = [selected_weight_row.to_dict()]
    final_model = sweep_models[selected_global_share]
    for train_percent in CATBOOST_TRAIN_SAMPLE_PERCENTS[1:]:
        final_model, metrics, summary = fit_catboost_experiment(
            train_percent, selected_global_share
        )
        experiment_metrics.append(metrics)
        learning_summaries.append(summary)
        print({key: round(value, 4) if isinstance(value, float) else value
               for key, value in summary.items()})

    learning_curve_pd = pd.DataFrame(learning_summaries).sort_values("train_percent")
    learning_curve_pd["delta_global_MAE_vs_previous"] = learning_curve_pd["global_MAE"].diff()
    learning_curve_pd["delta_delayed_MAE_vs_previous"] = learning_curve_pd["delayed_MAE"].diff()
    learning_curve_pd["delta_combined_vs_previous"] = learning_curve_pd["combined_MAE_score"].diff()
    learning_curve_pd.to_csv(LEARNING_CURVE_REPORT_PATH, index=False)
    CATBOOST_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    final_model.save_model(str(CATBOOST_MODEL_PATH))
    catboost_metrics_pd = pd.concat(experiment_metrics, ignore_index=True)
    display(learning_curve_pd)
    print({"model_path": str(CATBOOST_MODEL_PATH), "test_read": False})


{'model': 'catboost_g1.00_p01', 'train_percent': 1, 'train_rows': 24743, 'global_share': 1.0, 'delayed_row_weight': 1.0, 'best_iteration': 443, 'fit_seconds': 31.4887, 'global_MAE': np.float64(10.3698), 'delayed_MAE': np.float64(21.0299), 'punctual_MAE': np.float64(7.8147), 'severe_MAE': np.float64(76.8966), 'combined_MAE_score': np.float64(15.6998)}


{'model': 'catboost_g0.75_p01', 'train_percent': 1, 'train_rows': 24743, 'global_share': 0.75, 'delayed_row_weight': 2.7864, 'best_iteration': 431, 'fit_seconds': 28.6569, 'global_MAE': np.float64(10.5244), 'delayed_MAE': np.float64(21.0912), 'punctual_MAE': np.float64(7.9916), 'severe_MAE': np.float64(77.1121), 'combined_MAE_score': np.float64(15.8078)}


{'model': 'catboost_g0.65_p01', 'train_percent': 1, 'train_rows': 24743, 'global_share': 0.65, 'delayed_row_weight': 3.8857, 'best_iteration': 156, 'fit_seconds': 11.799, 'global_MAE': np.float64(10.6233), 'delayed_MAE': np.float64(21.0832), 'punctual_MAE': np.float64(8.1161), 'severe_MAE': np.float64(77.8527), 'combined_MAE_score': np.float64(15.8533)}


{'model': 'catboost_g0.50_p01', 'train_percent': 1, 'train_rows': 24743, 'global_share': 0.5, 'delayed_row_weight': 6.3591, 'best_iteration': 1, 'fit_seconds': 3.0144, 'global_MAE': np.float64(16.2251), 'delayed_MAE': np.float64(13.6812), 'punctual_MAE': np.float64(16.8349), 'severe_MAE': np.float64(72.2448), 'combined_MAE_score': np.float64(14.9531)}


,model,train_percent,train_rows,global_share,delayed_row_weight,best_iteration,fit_seconds,global_MAE,delayed_MAE,punctual_MAE,severe_MAE,combined_MAE_score,passes_global_guardrail
0,catboost_g1.00_p01,1,24743,1.00,1.000000,443,31.488657,10.369805,21.029858,7.814674,76.896594,15.699831,True
1,catboost_g0.75_p01,1,24743,0.75,2.786369,431,28.656949,10.524381,21.091227,7.991592,77.112127,15.807804,True
2,catboost_g0.65_p01,1,24743,0.65,3.885673,156,11.799018,10.623272,21.083231,8.116103,77.852714,15.853252,True
3,catboost_g0.50_p01,1,24743,0.50,6.359108,1,3.014388,16.225101,13.681188,16.834857,72.244808,14.953145,False


{'reference_global_MAE': np.float64(10.676), 'global_MAE_limit': np.float64(11.676), 'selected_global_share': 1.0}


{'model': 'catboost_g1.00_p05', 'train_percent': 5, 'train_rows': 123135, 'global_share': 1.0, 'delayed_row_weight': 1.0, 'best_iteration': 592, 'fit_seconds': 56.7318, 'global_MAE': np.float64(10.0588), 'delayed_MAE': np.float64(20.6014), 'punctual_MAE': np.float64(7.5318), 'severe_MAE': np.float64(75.8221), 'combined_MAE_score': np.float64(15.3301)}


{'model': 'catboost_g1.00_p10', 'train_percent': 10, 'train_rows': 245590, 'global_share': 1.0, 'delayed_row_weight': 1.0, 'best_iteration': 599, 'fit_seconds': 90.5931, 'global_MAE': np.float64(9.9836), 'delayed_MAE': np.float64(20.4836), 'punctual_MAE': np.float64(7.4668), 'severe_MAE': np.float64(75.5059), 'combined_MAE_score': np.float64(15.2336)}


,model,train_percent,train_rows,global_share,delayed_row_weight,best_iteration,fit_seconds,global_MAE,delayed_MAE,punctual_MAE,severe_MAE,combined_MAE_score,passes_global_guardrail,delta_global_MAE_vs_previous,delta_delayed_MAE_vs_previous,delta_combined_vs_previous
0,catboost_g1.00_p01,1,24743,1.0,1.0,443,31.488657,10.369805,21.029858,7.814674,76.896594,15.699831,True,NaN,NaN,NaN
1,catboost_g1.00_p05,5,123135,1.0,1.0,592,56.731832,10.058765,20.601351,7.531790,75.822080,15.330058,NaN,-0.311039,-0.428506,-0.369773
2,catboost_g1.00_p10,10,245590,1.0,1.0,599,90.593067,9.983605,20.483587,7.466842,75.505904,15.233596,NaN,-0.075160,-0.117764,-0.096462


{'model_path': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\models\\07_catboost_t60_selected_10pct.cbm', 'test_read': False}


## Combine with notebook 06 and rank by the frozen dual-MAE score

All candidates use identical validation rows. Notebook-06 models use 1% train, the historical
baseline uses full train, and the CatBoost learning curve reaches 10%. Therefore this is an
aligned validation comparison, not an equal-training-budget comparison. Ranking first enforces
the global-MAE guardrail and then minimizes the frozen dual-MAE score.


In [8]:
def build_selection_table(metrics):
    metrics = metrics.copy()
    metrics["candidate"] = metrics["model"] + " / " + metrics["numeric_variant"]
    global_rows = (
        metrics[metrics["segment"] == "all"]
        [["candidate", "training_scope", "MAE", "RMSE",
          "median_absolute_error", "p90_absolute_error"]]
        .rename(columns={
            "MAE": "global_MAE", "RMSE": "global_RMSE",
            "median_absolute_error": "global_median_absolute_error",
            "p90_absolute_error": "global_p90_absolute_error",
        })
    )
    delayed_rows = (
        metrics[metrics["segment"] == "delayed_>15"]
        [["candidate", "MAE", "RMSE", "median_absolute_error",
          "p90_absolute_error"]]
        .rename(columns={
            "MAE": "delayed_MAE", "RMSE": "delayed_RMSE",
            "median_absolute_error": "delayed_median_absolute_error",
            "p90_absolute_error": "delayed_p90_absolute_error",
        })
    )
    result = global_rows.merge(delayed_rows, on="candidate", validate="one_to_one")
    result["combined_MAE_score"] = (
        GLOBAL_MAE_WEIGHT * result["global_MAE"]
        + DELAYED_MAE_WEIGHT * result["delayed_MAE"]
    )
    result["passes_global_MAE_guardrail"] = result["global_MAE"] <= global_mae_limit
    return result.sort_values(
        ["passes_global_MAE_guardrail", "combined_MAE_score", "global_MAE"],
        ascending=[False, True, True],
    ).reset_index(drop=True)

if RUN_COMPARISON:
    model_metrics_pd = pd.read_csv(MODEL_REPORT_PATH)
    assert set(model_metrics_pd["status"]) == {"ok"}
    assert set(model_metrics_pd["validation_rows"].astype(int)) == {
        EXPECTED_VALIDATION_SAMPLE_ROWS
    }
    model_metrics_pd["training_scope"] = "deterministic_1pct_train"
    comparable_columns = [
        "model", "numeric_variant", "status", "training_scope",
        "segment", "rows", "MAE", "RMSE",
        "median_absolute_error", "p90_absolute_error",
    ]
    all_metrics_pd = pd.concat([
        model_metrics_pd[comparable_columns],
        catboost_metrics_pd[comparable_columns],
        baseline_metrics_pd[comparable_columns],
    ], ignore_index=True)
    selection_table = build_selection_table(all_metrics_pd)
    REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    selection_table.to_csv(REPORT_PATH, index=False)
    display(selection_table)
    print({
        "selected_with_global_MAE_guardrail": selection_table.iloc[0]["candidate"],
        "report": str(REPORT_PATH),
        "test_read": False,
    })
    baseline_predictions.unpersist()
    spark.stop()
else:
    print("Set RUN_COMPARISON=True and run all cells.")


,candidate,training_scope,global_MAE,global_RMSE,global_median_absolute_error,global_p90_absolute_error,delayed_MAE,delayed_RMSE,delayed_median_absolute_error,delayed_p90_absolute_error,combined_MAE_score,passes_global_MAE_guardrail
0,linear_regression / original,deterministic_1pct_train,10.871172,15.916364,8.200432,21.883651,19.267681,27.252181,14.796133,37.475823,15.069427,True
1,linear_regression / log,deterministic_1pct_train,10.875577,15.926486,8.215018,21.934566,19.322786,27.295365,14.820095,37.502742,15.099181,True
2,linear_regression / yeo_johnson,deterministic_1pct_train,10.874277,15.927735,8.211066,21.904444,19.338263,27.316828,14.787736,37.429120,15.106270,True
3,catboost_g1.00_p10 / native_categorical_t60,deterministic_10pct_train,9.983605,15.210577,7.283929,20.276621,20.483587,28.167711,15.957293,38.389129,15.233596,True
4,catboost_g1.00_p05 / native_categorical_t60,deterministic_5pct_train,10.058765,15.293395,7.357490,20.434142,20.601351,28.241610,15.917028,38.628064,15.330058,True
5,historical_route_airline_fallback / not_applic...,full_train,9.938292,15.270555,7.116667,20.450000,21.108836,28.662692,16.683333,39.200000,15.523564,True
6,catboost_g1.00_p01 / native_categorical_t60,deterministic_1pct_train,10.369805,15.615384,7.654085,21.003379,21.029858,28.566233,16.088153,39.190054,15.699831,True
7,catboost_g0.75_p01 / native_categorical_t60,deterministic_1pct_train,10.524381,15.764209,7.805905,21.468329,21.091227,28.624611,16.248920,39.575799,15.807804,True
8,catboost_g0.65_p01 / native_categorical_t60,deterministic_1pct_train,10.623272,15.856930,7.901879,21.544303,21.083231,28.647616,16.231096,39.388176,15.853252,True
9,xgboost / original,deterministic_1pct_train,10.676004,16.069822,7.762943,22.072706,23.199031,30.091729,18.623354,40.710914,16.937518,True


{'selected_with_global_MAE_guardrail': 'linear_regression / original', 'report': 'C:\\Users\\celti\\OneDrive - Universidade de Santiago de Compostela\\Verano\\ML_flights_project\\reports\\07_aligned_validation_comparison.csv', 'test_read': False}


## Interpretation guardrails

- Reject a CatBoost weighting if global MAE exceeds the best notebook-06 value by >1 minute.
- Among admissible candidates, select by the agreed equal-weight combined score.
- Do not infer that the delayed segment is observable at T-60; it is an evaluation slice.
- Do not read test or retune the historical thresholds in this notebook.
- Do not claim equal training budgets: baseline uses full train, notebook 06 uses 1%, and the
  CatBoost learning curve uses 1%, 5% and 10%.
- CatBoost uses native categories and weighted MAE; it does not yet use the historical baseline
  as a feature, because train rows need expanding historical encodings first.
- Previously observed flight outcomes at T-60 are reserved for a later temporal-feature step.
